## Net generators

## Tipus de xarxes

### No geomètriques

- Erdos-Ranyi (no es scale-free, clustering baix)
- Barabasi-Albert (scale-free, clustering baix)
- Configuracional (clustering = 0)

### Geomètriques

- $\mathbb{S}^1\mathbb{H}^2$


## Equivalència de paràmetres

| Característica                                  | ER                      | BA                                   | Configuracional                                   | S1 / H2                 |
| ----------------------------------------------- | ----------------------- | ------------------------------------ | ------------------------------------------------- | ----------------------- |
| $N$                                             | $N$                     | $N$                                  | $N$                                               | $N$                     |
| Grau mig $\langle k \rangle$                    | $p=\frac{⟨k⟩}{N−1}$     | $m=\frac{⟨k⟩}{2}$​ (ha de ser enter) | En funció de la sequencia o distribució de graus. | $\langle k \rangle$     |
| Exponent de la distribució $\gamma$ (si aplica) | -                       | Fixat, $\gamma = 3$                  | Escollir $P(k) = k^{-\gamma}$                     | $\gamma$                |
| Clustering $C$                                  | Molt baix, no ajustable | Molt baix, no ajustable              | 0, no ajustable                                   | En funció de la $\beta$ |


In [ ]:

g = 2.1
n = 1000
b = 2.1
k = 20
m = int(k/2)
p = k/(n-1)
seed = 12345


working_folder = f'./generated-nets'
s1h2_working_folder = f'{working_folder}/s1h2'
er_working_folder = f'{working_folder}/er'
ba_working_folder = f'{working_folder}/ba'
conf_working_folder = f'{working_folder}/config'

er_file = f'{er_working_folder}/er-n={n}-k={k}'
ba_file = f'{ba_working_folder}/ba-n={n}-k={k}'
conf_file = f'{conf_working_folder}/conf-n={n}-k={k}-g={g}'
s1h2_file = f'{s1h2_working_folder}/s1h2-n={n}-k={k}-g={g}-b={b}'

er_edges_file = f'{er_file}.edge'
ba_edges_file = f'{ba_file}.edge'
conf_edges_file = f'{conf_file}.edge'
s1h2_edges_file = f'{s1h2_file}.edge'


er_coords_file = f'{er_file}.inf_coord'
ba_coords_file = f'{ba_file}.inf_coord'
conf_coords_file = f'{conf_file}.inf_coord'
s1h2_coords_file = f'{s1h2_file}.inf_coord'


In [ ]:
import re
import networkx as nx
from pathlib import Path
def generate_powerlaw_sequence_with_avg_k(n, gamma, target_k, k_max=100, tolerance=0.1, seed=12345):
    """
    Genera una secuencia de grados con γ fijo que se aproxime a <k> objetivo.
    """
    import numpy as np
    # Buscar k_min que dé el <k> deseado
    k_min = 1
    best_seq = None
    best_avg = 0
    rng = np.random.RandomState(seed)
    for k_min_candidate in range(1, 20):
        # Generar grados con este k_min
        degrees = []
        while len(degrees) < n:
            # Muestreo por transformada inversa
            r = rng.random()
            k = k_min_candidate * (1 - r) ** (-1/(gamma-1))
            if k <= k_max:
                degrees.append(int(np.floor(k)))
        
        # Ajustar para que la suma sea par (requisito del modelo de configuración)
        if sum(degrees) % 2 != 0:
            degrees[0] += 1
        
        current_avg = np.mean(degrees)
        
        if abs(current_avg - target_k) < abs(best_avg - target_k):
            best_avg = current_avg
            best_seq = degrees
        
        if abs(current_avg - target_k) < tolerance:
            return degrees
    
    return best_seq

Gba = nx.barabasi_albert_graph(n, m, seed=seed)
sequence = generate_powerlaw_sequence_with_avg_k(n, gamma=g, target_k=k, seed=seed, tolerance=0.001)
Gconf = nx.configuration_model(sequence)
Ger = nx.erdos_renyi_graph(n, p, seed=seed)

! mkdir -p {s1h2_working_folder}
! mkdir -p {er_working_folder}
! mkdir -p {ba_working_folder}
! mkdir -p {conf_working_folder}

nx.write_edgelist(Gba, ba_edges_file, data=False)
nx.write_edgelist(Ger, er_edges_file, data=False)
nx.write_edgelist(Gconf, conf_edges_file, data=False)
! ./tools/genSD -d 1 -n {n} -g {g} -k {k} -b {b} -s {seed} -o {s1h2_file}
# remove v from node name
t = Path(s1h2_edges_file).read_text()
t = re.sub(r'v(\d)', r'\1', t)
Path(s1h2_edges_file).write_text(t)

Gs1h2 = nx.read_edgelist(s1h2_edges_file)

# Generació

In [ ]:
! ./tools/mercator -o {s1h2_file} -b {b} -s {seed} {s1h2_edges_file}
! ./tools/mercator -o {er_file} -b {b} -s {seed} {er_edges_file}
! ./tools/mercator -o {ba_file} -b {b} -s {seed} {ba_edges_file}
! ./tools/mercator -o {conf_file} -b {b} -s {seed} {conf_edges_file}


In [ ]:
import pipeline.data as data

Gba, dfba, paramsba = data.read_hyperbolic_data(ba_coords_file, ba_edges_file)
Ger, dfer, paramser = data.read_hyperbolic_data(er_coords_file, er_edges_file)
Gconf, dfconf, paramsconf = data.read_hyperbolic_data(conf_coords_file, conf_edges_file)
Gs1h2 , dfs1h2, paramss1h2 = data.read_hyperbolic_data(s1h2_coords_file, s1h2_edges_file)

#### Comparació diferents nodes inicials

In [ ]:
import pipeline.hyperbolic as hyp

coords = (
    dfba.set_index('Vertex')
    [['Disc.Radius', 'Inf.Theta']]
    .to_dict('index')
)

hyp_distances = {}
Gk = {}
for u, v in Gba.edges():
    
    r1 = coords[u]['Disc.Radius']
    t1 = coords[u]['Inf.Theta']

    r2 = coords[v]['Disc.Radius']
    t2 = coords[v]['Inf.Theta']

    d = hyp.hyperbolic_distance_og_thetas(r1, t1, r2, t2)

    Gba[u][v]['dist'] = d

In [ ]:
import numpy as np
import pandas as pd
R = 2 * np.log(paramsba['nb. vertices']/(paramsba['mu']*np.pi*paramsba['kappa_min']**2))


edges = pd.DataFrame(Gba.edges, columns=['a', 'b'])
edges = pd.merge(edges, dfba[['Vertex', 'Disc.Radius', 'Inf.Theta']], left_on='a', right_on='Vertex', suffixes=('_a', '_b'))
edges = pd.merge(edges, dfba[['Vertex', 'Disc.Radius', 'Inf.Theta']], left_on='b', right_on='Vertex', suffixes=('_a', '_b'))
edges['Theta_Dif'] = np.pi - np.abs(np.pi - np.abs(edges['Inf.Theta_a']-edges['Inf.Theta_b']))

edges['Distance'] = np.where(edges['Theta_Dif'] == 0, 
                            np.abs(edges['Disc.Radius_a']- edges['Disc.Radius_b']), 
                            hyp.hyperbolic_distance_og(edges['Disc.Radius_a'], edges['Disc.Radius_b'], edges['Theta_Dif']))



for n in range(10):
    c = -(n+1)/10
    edges['Epidemic_Func'] = hyp.link_probability_og(edges['Distance'], R, c)

    avg_epidemic_func = np.average(edges['Epidemic_Func'])

    edges['Weight_Multiplier'] = edges['Epidemic_Func']/avg_epidemic_func
    edges.to_csv(f"{ba_edges_file}_weight_-{n+1}x10^-1", sep='\t', header=False, index=False, columns=['a', 'b', 'Weight_Multiplier'])

In [ ]:
events_sn = {}
targeted_epidemic_folder = f'{working_folder}/targeted-epidemics'
! mkdir -p {targeted_epidemic_folder} 
output_batch_file = f"{targeted_epidemic_folder}/batch_targeted_sir.txt"

seed_base = 42075
seeds = 15
i_rate = 1
r_rate = 1
limit_time=1E10
model_type=2 # sir
c='-1x10^-1'
weight_file = f'{ba_edges_file}_weight_{c}'
weighted=True
n=0

In [ ]:



with open(output_batch_file, "w") as f:
    f.write("# infection_rate  recovery_rate  seed  limit_time  model_type start_node\n")
    for v in dfba['Vertex']:
        for s in range(seeds):
        # Escribir línea con 5 valores (start_node opcional omitido)
            f.write(f"{i_rate:.6f} {r_rate:.6f} {seed_base+s} {limit_time:.1f} {model_type} {v}\n")
            n = n+1

print(f"Archivo batch generado: {output_batch_file} con {n} simulaciones.")

In [ ]:
! ./tools/epidemics -b {output_batch_file} -o {targeted_epidemic_folder} -ev -w {weight_file}

In [ ]:
from tqdm import tqdm
deg = Gba.degree()
events_sn = {}
seed_base = 42075
seeds = 15
for n, k in tqdm(deg):
    for s in range(seeds):
        i_rate = 1
        r_rate = 1
        model='SIR'
        c='-1x10^-1'
        weight_file = f'{ba_edges_file}_weight_{c}'
        weighted=True
        events_file = f'{targeted_epidemic_folder}/{data.build_events_filename(ba_edges_file, weighted, model, i_rate, r_rate, seed_base+s, int(n))}'
        events_sn[(n, s)] = data.read_events_data(events_file)

In [ ]:
import numpy as np
from tqdm import tqdm
import pipeline.animation as anim

# ------------------------------------------------------------
# 1. Preprocesamiento: posiciones de los nodos
# ------------------------------------------------------------
vertices = df['Vertex'].values
num_nodes = len(vertices)
vertex_to_idx = {v: i for i, v in enumerate(vertices)}
radii = df['Disc.Radius'].values
thetas = df['Inf.Theta'].values

# ------------------------------------------------------------
# 3. Precalcular matriz de distancias hiperbólicas (N x N)
#    Solo si N es razonable (ej. < 5000). Si no, habrá que
#    calcular distancias sobre la marcha pero evitando recalcular snapshots.
# ------------------------------------------------------------
print("Precalculando matriz de distancias hiperbólicas...")
# Función vectorizada que calcula distancias entre dos conjuntos de puntos
# (puedes adaptar tu hyp.hyperbolic_distance_og_thetas para que trabaje con broadcasting)
def hyperbolic_distance_matrix(r1, theta1, r2, theta2):
    # Implementación típica de distancia hiperbólica en el disco de Poincaré
    # d = arcosh(cosh(r1)*cosh(r2) - sinh(r1)*sinh(r2)*cos(theta1 - theta2))
    # Devuelve matriz de tamaño (len(r1), len(r2))
    cosh_r1 = np.cosh(r1)
    cosh_r2 = np.cosh(r2)
    sinh_r1 = np.sinh(r1)
    sinh_r2 = np.sinh(r2)
    delta_theta = np.abs(theta1[:, None] - theta2[None, :])
    cos_delta = np.cos(delta_theta)
    arg = cosh_r1[:, None] * cosh_r2[None, :] - sinh_r1[:, None] * sinh_r2[None, :] * cos_delta
    # Evitar NaN por errores de redondeo
    arg = np.clip(arg, 1.0, None)   # fuerza valores menores a 1 a ser exactamente 1
    return np.arccosh(arg)  
D = hyperbolic_distance_matrix(radii, thetas, radii, thetas)  # D[i,j] = distancia entre nodo i y nodo j


In [ ]:
# 3. Tiempos
xs = np.linspace(0, 2, 1000)

# 4. Diccionario para almacenar resultados por nodo
avg_hyps = {}
err_hyps = {}
avg_hyps_maxs = {}

def compute_node(nk):
    n, k = nk
    idx = vertex_to_idx[n]
    avg_over_t = []
    std_over_t = []
    max_over_t = []
    last_snap_by_seed = [None] * seeds

    for t in xs:
        dist_means = []
        dist_stds = []
        dist_maxs = []
        for s in range(seeds):
            events = events_sn.get((n, s), [])
            if len(events) == 3:
                continue
            # La función get_snapshot se llama aquí
            snap = anim.get_snapshot(df, events, t, last_snap_by_seed[s])
            last_snap_by_seed[s] = snap
            infected_verts = snap['infected']
            infected_idxs = np.array([vertex_to_idx[v] for v in infected_verts if v in vertex_to_idx])
            if len(infected_idxs) == 0:
                dist_means.append(0.0)
                dist_stds.append(0.0)
                dist_maxs.append(0.0)
            else:
                dists = D[idx, infected_idxs]
                dist_means.append(np.mean(dists))
                dist_stds.append(np.std(dists))
                dist_maxs.append(np.max(dists))
        avg_over_t.append(np.mean(dist_means) if dist_means else np.nan)
        std_over_t.append(np.mean(dist_stds) if dist_stds else np.nan)
        max_over_t.append(np.mean(dist_maxs) if dist_maxs else np.nan)

    return n, avg_over_t, std_over_t, max_over_t

import multiprocessing as mp
# 1. Cargar / definir todos los datos grandes (df, vertex_to_idx, D, xs, seeds, events_sn, anim)
xs = np.linspace(0, 0.7, 1000)
# ... (código de inicialización)

# 2. Forzar uso de fork (por si acaso)
mp.set_start_method('fork', force=True)

# 3. Lanzar Pool
with mp.Pool(processes=8) as pool:
    resultados = list(tqdm(pool.imap_unordered(compute_node, list(G.degree())),
                            total=G.number_of_nodes(),
                            desc="Procesando nodos"))

# 4. Acumular resultados
avg_hyps = {}
err_hyps = {}
avg_hyps_maxs = {}
for n, avg, std, mx in resultados:
    avg_hyps[n] = avg
    err_hyps[n] = std
    avg_hyps_maxs[n] = mx

In [ ]:
results_folder = f'{targeted_epidemic_folder}/results'
! mkdir -p {results_folder} 
for v, k in G.degree:
    filename = f'{results_folder}/n={v}.dat'
    data = np.transpose([xs, avg_hyps[v]])
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f'# DATA\n')
        f.write(f'# sn={v}\n')
        f.write(f'# k={k}\n')
        f.write(f'#\n')
        f.write(f'#\n')
        f.write('# t, <d_hyp>\n')
        for x, avg_hyp in data:
            f.write(f'{x}\t{avg_hyp}\n')

In [ ]:
targeted_epidemic_folder = f'{working_folder}/targeted-epidemics'
results_folder = f'{targeted_epidemic_folder}/results'

def load_velocities_result(results_folder, vertex):
    xs = None
    avg_hyps = {}
    for v in vertex:
        filename = f'{results_folder}/n={v}.dat'
        df_v = pd.read_csv(filename, sep=r'\s+', comment='#', names=['t', 'd'])
        xs = df_v['t']
        avg_hyps[v] = df_v['d']
    return xs, avg_hyps

def load_velocities_by_degree(results_folder, vertex):
    xs = None
    avg_hyps = {}
    for v in vertex:
        filename = f'{results_folder}/n={v}.dat'
        df_v = pd.read_csv(filename, sep=r'\s+', comment='#', names=['t', 'd'])
        xs = df_v['t']
        avg_hyps[v] = df_v['d']
    return xs, avg_hyps

xs, avg_hyps = load_velocities_result(results_folder, df['Vertex'])

In [ ]:
from scipy.optimize import curve_fit
import uncertainties as un


fig, ax = plt.subplots(dpi=300)

curvs = []

selected = []
deg = G.degree()


maxdeg, mindeg = max(deg, key=lambda x: x[1])[1], min(deg, key=lambda x: x[1])[1]
deg_space = np.round(np.linspace(mindeg, maxdeg, 8)[1:-1])
deg_space

sort = sorted(G.degree(), key=lambda x:x[1])
deg_i = 0

for i in range(len(sort)):
    k, deg = sort[i]
    if (deg > deg_space[deg_i]):
        selected.append(sort[i])
        deg_i += 1
        if (deg_i >= len(deg_space)):
            break

# ... (código anterior: selección de nodos, etc.)

for i, (n, k_deg) in enumerate(selected):   # k_deg es el grado (renombrado para no confundir con k de sigmoid)
    ax.scatter(xs, avg_hyps[n], s=1, label=fr'$k_\star = {k_deg}$', c=plt.cm.tab20.colors[2*i])
    
ax.legend()
ax.set_xlabel(r"$t$ (\unit{\second})")
ax.set_ylabel(r"$\langle d_\text{hyp}(x_i, x_\star)\rangle$")
plt.show()

In [ ]:
from scipy.optimize import curve_fit
import uncertainties as un


fig, ax = plt.subplots(dpi=300)

curvs = []

selected = []
deg = G.degree()


maxdeg, mindeg = max(deg, key=lambda x: x[1])[1], min(deg, key=lambda x: x[1])[1]
deg_space = np.round(np.linspace(mindeg, maxdeg, 8)[1:-1])
deg_space

sort = sorted(G.degree(), key=lambda x:x[1])
deg_i = 0

for i in range(len(sort)):
    k, deg = sort[i]
    if (deg > deg_space[deg_i]):
        selected.append(sort[i])
        deg_i += 1
        if (deg_i >= len(deg_space)):
            break

def linear_sigmoid(x, L, d, x0, k):
    return (L) / (1 + np.exp(-k * (x - x0))) + d

def gompertz(x, L, d, x0, k):
    return L*np.exp(-np.exp(-k*(x-x0)))+d

def hill(x, L, a, d, K, k):
    t = x+a
    return L*t**k/(t**k+K**k)+d

# ... (código anterior: selección de nodos, etc.)

for i, (n, k_deg) in enumerate(selected):   # k_deg es el grado (renombrado para no confundir con k de sigmoid)
    ax.scatter(xs, avg_hyps[n], s=1, label=fr'$k_\star = {k_deg}$', c=plt.cm.tab20.colors[2*i+1])
    
    y = avg_hyps[n]
    # initial values
    d0 = np.percentile(y, 5)

# --- remove drift ---
    y_corr = y 

    # --- amplitude ---
    L0 = np.percentile(y_corr, 95) - d0

    # --- half-saturation ---
    y_h = y - (d0)
    half = L0/2
    idx = np.argmin(np.abs(y_h - half))
    K0 = xs[idx]

    # --- shift ---
    a0 = 0.05*K0

    # --- steepness ---
    dy = np.gradient(y_corr, xs)
    v0 = np.max(dy)
    k0 = np.clip(4*K0*v0/L0, 0.5, 10)

    p0 = [L0, a0, d0, K0, k0]
    
    popt, pcov = curve_fit(
        hill,
        xs,
        y,
        p0=p0,
        # bounds=bounds,
        maxfev=20000
    )
    perr = np.sqrt(np.diag(pcov))
    y_fit = hill(xs, *popt)
    ax.plot(xs, y_fit, "--")
    print(popt)
    curvs.append(un.ufloat(popt[-1], perr[-1]))   # guardamos 'k' (pendiente de la sigmoide)
ax.legend()
ax.set_xlabel(r"$t$ (\unit{\second})")
ax.set_ylabel(r"$\langle d_\text{hyp}(x_i, x_\star)\rangle$")
plt.show()

In [ ]:
from scipy.optimize import curve_fit
import uncertainties as un


df_deg = pd.DataFrame([{'Vertex':v, 'Degree':k} for v, k in G.degree()])
vels = []
err_vels = []
for k in (sorted(set(df_deg['Degree'].astype(int)))):
    vel = []
    for n in tqdm(df_deg[df_deg['Degree'] == k]['Vertex']):
        # p0 = [max(avg_hyps[n]), np.median(xs),1] # this is an mandatory initial guess
        # popt, pcov = curve_fit(sigmoid, xs, avg_hyps[n], p0, method='dogbox', maxfev=10000)
        # vel.append(popt[2]*popt[0]/4)
        y = avg_hyps[n]
        # initial values
        d0 = np.percentile(y, 5)

    # --- remove drift ---
        y_corr = y 

        # --- amplitude ---
        L0 = np.percentile(y_corr, 95) - d0

        # --- half-saturation ---
        y_h = y - (d0)
        half = L0/2
        idx = np.argmin(np.abs(y_h - half))
        K0 = xs[idx]

        # --- shift ---
        a0 = 0.05*K0

        # --- steepness ---
        dy = np.gradient(y_corr, xs)
        v0 = np.max(dy)
        k0 = np.clip(4*K0*v0/L0, 0.5, 10)

        p0 = [L0, a0, d0, K0, k0]
        
        popt, pcov = curve_fit(
            hill,
            xs,
            y,
            p0=p0,
            # bounds=bounds,
            maxfev=20000
        )
        L, a, d, lambd, k = popt

        vmax = (L*(k+1)**2/(4*k*lambd))*((k-1)/(k+1))**((k-1)/k)
        # vmax = L*k/(4*lambd)
        vel.append(vmax)
 
    vels.append(np.mean(vel))
    err_vels.append(np.std(vel))

In [ ]:
vels

In [ ]:
fig, ax = plt.subplots(dpi=300)
x_degs = np.array(sorted(set(df_deg['Degree'].astype(int))))
vels = np.array(vels)

ax.errorbar(x_degs, vels, 
            yerr=err_vels,
            fmt='o', markersize=1, elinewidth=0.5)

mask = vels < 4000

x_fit = x_degs[mask]
y_fit = vels[mask]


coefs, errores, r2, cov_mat = ajuste_polinomial_con_estadisticas_v2(x_fit, y_fit, 1)

x_fit_plot = np.linspace(0, 200, 100)
y_fit_plot = coefs[1]*x_fit_plot + coefs[0]

ax.plot(x_fit_plot, y_fit_plot, '--', linewidth=1)
print(r2)
ax.set_xlabel(r'$k_\star$')
ax.set_ylabel(r'$v_\text{max}$ (\unit{\per\second})')
ax.set_ylim(0, 4000)
ax.set_xlim(0, 200)
plt.show()


In [ ]:
from glob import glob

degs_vels = {}

folders = [
    "pipeline-output/out-n1000-g=2.1-b=2.1-k=20-s=12345/targeted-epidemics/results/*",
    "pipeline-output/out-n1000-g=2.1-b=2.1-k=20-s=12346/targeted-epidemics/results/*",
]

for folder in folders:
    for filename in tqdm(glob(folder)):
        deg_dist_df = pd.read_csv(filename, sep=r'\s+', names=['t', 'dist'], comment='#')

        params = {}
        with open(filename, 'r') as f:
            for line in f:
                if line.startswith('#') and '=' in line:
                    parts = line.strip('# ').split('=')
                    if len(parts) == 2:
                        key = parts[0].strip()
                        if (key.startswith('-')):
                            key = key[1:].strip()
                        try:
                            params[key] = float(parts[1].strip())
                        except ValueError:
                            params[key] = parts[1].strip()
        y = deg_dist_df['dist']
        xs = deg_dist_df['t']
            # initial values
        d0 = np.percentile(y, 5)

    # --- remove drift ---
        y_corr = y 

        # --- amplitude ---
        L0 = np.percentile(y_corr, 95) - d0

        # --- half-saturation ---
        y_h = y - (d0)
        half = L0/2
        idx = np.argmin(np.abs(y_h - half))
        K0 = xs[idx]

        # --- shift ---
        a0 = 0.05*K0

        # --- steepness ---
        dy = np.gradient(y_corr, xs)
        v0 = np.max(dy)
        k0 = np.clip(4*K0*v0/L0, 0.5, 10)

        p0 = [L0, a0, d0, K0, k0]
        
        popt, pcov = curve_fit(
            hill,
            xs,
            y,
            p0=p0,
            # bounds=bounds,
            maxfev=20000
        )
        L, a, d, lambd, k = popt

        vmax = (L*(k+1)**2/(4*k*lambd))*((k-1)/(k+1))**((k-1)/k)
        # vmax = L*k/(4*lambd)
        if params['k'] not in degs_vels:
            degs_vels[params['k']] = []
        degs_vels[params['k']].append(vmax)
    
    


In [ ]:
x_degs = np.array(sorted(degs_vels))
vels = np.array([np.mean(degs_vels[deg]) for deg in x_degs])
err_vels = np.array([np.std(degs_vels[deg]) for deg in x_degs])
fig, ax = plt.subplots(dpi=300)

ax.errorbar(x_degs, vels, 
            yerr=err_vels,
            fmt='o', markersize=1, elinewidth=0.5)

mask = vels < 4000

x_fit = x_degs[mask]
y_fit = vels[mask]


coefs, errores, r2, cov_mat = ajuste_polinomial_con_estadisticas_v2(x_fit, y_fit, 1)

x_fit_plot = np.linspace(0, 200, 100)
y_fit_plot = coefs[1]*x_fit_plot + coefs[0]

ax.plot(x_fit_plot, y_fit_plot, '--', linewidth=1)
print(r2)
ax.set_xlabel(r'$k_\star$')
ax.set_ylabel(r'$v_\text{max}$ (\unit{\per\second})')
ax.set_ylim(0, 4000)
ax.set_xlim(0, 200)
plt.show()
